<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Method choice: Random Forest classifier

I will use a Random Forest classifier to estimate whether a visible content item will have below-benchmark CTR in March, using only February signals.

A tree ensemble fits this lane because impressions, search position, and engagement can have non-linear relationships with future CTR performance. I will handle missing February values explicitly inside the training pipeline and keep the feature set limited to information available before the March outcome window.

The model is decision-support: it creates a ranked review queue. It does not prove that a page change will cause more clicks.

In [6]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{hf_token}')"
)

FEB = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/*.parquet"
)

MARCH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

model_df = con.execute(f"""
    WITH february_features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS feb_impressions,
            AVG(NULLIF(gsc_avg_position, 0)) AS feb_avg_position,
            SUM(
                CASE WHEN ga4_data_available IS TRUE
                THEN scroll_events END
            ) AS feb_scroll_events,
            SUM(
                CASE WHEN ga4_data_available IS TRUE
                THEN sessions_social END
            ) AS feb_sessions_social,
            COUNT(*) AS feb_observed_days,
            MAX(
                CASE WHEN ga4_data_available IS TRUE
                THEN 1 ELSE 0 END
            ) AS has_ga4_data
        FROM read_parquet('{FEB}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    march_outcomes AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            SUM(gsc_clicks)::DOUBLE
                / NULLIF(SUM(gsc_impressions), 0) AS march_ctr,
            AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position
        FROM read_parquet('{MARCH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    march_bucketed AS (
        SELECT *,
            CASE
                WHEN march_avg_position <= 3 THEN '1-3'
                WHEN march_avg_position <= 10 THEN '4-10'
                WHEN march_avg_position <= 20 THEN '11-20'
                WHEN march_avg_position <= 50 THEN '21-50'
                ELSE '51+'
            END AS march_position_bucket
        FROM march_outcomes
        WHERE march_impressions >= 100
          AND march_avg_position IS NOT NULL
    ),
    labelled_march AS (
        SELECT *,
            MEDIAN(march_ctr) OVER (
                PARTITION BY march_position_bucket
            ) AS march_bucket_median_ctr
        FROM march_bucketed
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.feb_impressions,
        f.feb_avg_position,
        f.feb_scroll_events,
        f.feb_sessions_social,
        f.feb_observed_days,
        f.has_ga4_data,
        CASE
            WHEN f.feb_avg_position <= 3 THEN '1-3'
            WHEN f.feb_avg_position <= 10 THEN '4-10'
            WHEN f.feb_avg_position <= 20 THEN '11-20'
            WHEN f.feb_avg_position <= 50 THEN '21-50'
            ELSE 'missing_or_51+'
        END AS feb_position_bucket,
        m.march_position_bucket,
        m.march_ctr,
        m.march_bucket_median_ctr,
        CASE
            WHEN m.march_ctr < m.march_bucket_median_ctr THEN 1
            ELSE 0
        END AS is_march_below_bucket_median_ctr
    FROM february_features AS f
    INNER JOIN labelled_march AS m
        USING (client_hash_id, content_hash_id)
    WHERE f.feb_impressions >= 100
      AND f.feb_avg_position IS NOT NULL
""").df()

print("=== ML-08 modelling dataset ===")
print(f"Rows: {len(model_df):,}")
print(
    "March below-benchmark target rate: "
    f"{model_df['is_march_below_bucket_median_ctr'].mean():.3f}"
)
print(f"Unique pseudonymized clients: {model_df['client_hash_id'].nunique():,}")

display(model_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== ML-08 modelling dataset ===
Rows: 73,481
March below-benchmark target rate: 0.388
Unique pseudonymized clients: 33


,client_hash_id,content_hash_id,feb_impressions,feb_avg_position,feb_scroll_events,feb_sessions_social,feb_observed_days,has_ga4_data,feb_position_bucket,march_position_bucket,march_ctr,march_bucket_median_ctr,is_march_below_bucket_median_ctr
0,client_9958f0a7ae1df715,content_6c474bcf2783d986,265.0,20.583427,NaN,NaN,28,0,21-50,21-50,0.000000,0.0,0
1,client_62f4a7e64f5e0096,content_a98f0da6765e3fda,259.0,4.885870,NaN,NaN,28,0,4-10,21-50,0.000000,0.0,0
2,client_e5c2aa26a8598242,content_d7255a15ed9e3446,878.0,21.367225,NaN,NaN,10,0,21-50,21-50,0.003526,0.0,0
3,client_e5c2aa26a8598242,content_9c50712f53a3df7c,405.0,22.708045,NaN,NaN,10,0,21-50,21-50,0.000000,0.0,0
4,client_9958f0a7ae1df715,content_a22ef2f4631595f1,318.0,36.310620,NaN,NaN,28,0,21-50,21-50,0.000000,0.0,0
5,client_9958f0a7ae1df715,content_6abbb579b216aa96,322.0,29.904746,2.0,0.0,28,1,21-50,21-50,0.005000,0.0,0
6,client_9958f0a7ae1df715,content_c6138fac1cb3c601,382.0,29.162985,2.0,0.0,28,1,21-50,21-50,0.000000,0.0,0
7,client_9958f0a7ae1df715,content_ea7557b951a6aa41,200.0,17.785363,3.0,0.0,28,1,11-20,21-50,0.000000,0.0,0
8,client_9958f0a7ae1df715,content_8a2193f569f8d127,139.0,21.886127,0.0,0.0,27,1,21-50,21-50,0.000000,0.0,0
9,client_9958f0a7ae1df715,content_3bd4c24c3160b8eb,343.0,16.566308,0.0,0.0,28,1,11-20,21-50,0.008696,0.0,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design: grouped client holdout

I will hold out whole pseudonymized clients for validation. All items from a client belong either to training or validation, never both.

This is more honest than a random row split because client-specific search patterns could otherwise appear in both datasets and make performance look better than it would on a new client. February features are available before the March target window, so the split is also time-aware.

The transparent baseline uses only February visibility: it prioritizes items in February positions 4–20 by their February impression volume. It is a simple review-queue rule, evaluated on the same held-out clients and the same March target as the model.

In [7]:
from sklearn.model_selection import StratifiedGroupKFold

feature_columns = [
    "feb_impressions",
    "feb_avg_position",
    "feb_scroll_events",
    "feb_sessions_social",
    "feb_observed_days",
    "has_ga4_data",
    "feb_position_bucket",
]

target_column = "is_march_below_bucket_median_ctr"

X = model_df[feature_columns].copy()
y = model_df[target_column].copy()
groups = model_df["client_hash_id"].copy()

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

candidate_splits = list(splitter.split(X, y, groups))

# Select the client-held-out fold whose target rate is closest to the full dataset.
overall_target_rate = y.mean()

train_index, validation_index = min(
    candidate_splits,
    key=lambda split: abs(y.iloc[split[1]].mean() - overall_target_rate),
)

X_train = X.iloc[train_index].copy()
X_val = X.iloc[validation_index].copy()
y_train = y.iloc[train_index].copy()
y_val = y.iloc[validation_index].copy()

train_clients = set(groups.iloc[train_index])
validation_clients = set(groups.iloc[validation_index])

assert train_clients.isdisjoint(validation_clients)

baseline_val_score = np.where(
    X_val["feb_avg_position"].between(4, 20),
    np.log1p(X_val["feb_impressions"]),
    0.0,
)

print("=== Stratified grouped client split ===")
print(f"Overall target rate: {overall_target_rate:.3f}")
print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_val):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Validation clients: {len(validation_clients)}")
print(f"Client overlap: {len(train_clients.intersection(validation_clients))}")
print(f"Training target rate: {y_train.mean():.3f}")
print(f"Validation target rate: {y_val.mean():.3f}")

print("\n=== February-only baseline ===")
print(
    "Rule: rank February positions 4–20 by February impression volume; "
    "other items receive score 0."
)
print(
    f"Validation items with a positive baseline score: "
    f"{(baseline_val_score > 0).sum():,}"
)

=== Stratified grouped client split ===
Overall target rate: 0.388
Training rows: 61,001
Validation rows: 12,480
Training clients: 21
Validation clients: 12
Client overlap: 0
Training target rate: 0.391
Validation target rate: 0.372

=== February-only baseline ===
Rule: rank February positions 4–20 by February impression volume; other items receive score 0.
Validation items with a positive baseline score: 8,993


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model and baseline comparison

The primary metric is **Precision@20** because the decision is a small human review queue: among the 20 highest-ranked held-out candidates, how many are actually below their March position-bucket CTR benchmark?

I will also report Average Precision and ROC-AUC as secondary ranking diagnostics. Both the Random Forest and the transparent February-only baseline are evaluated on the same held-out clients, using the same March target.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "feb_impressions",
    "feb_avg_position",
    "feb_scroll_events",
    "feb_sessions_social",
    "feb_observed_days",
    "has_ga4_data",
]

categorical_features = [
    "feb_position_bucket",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "impute",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True,
                        ),
                    ),
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "impute",
                        SimpleImputer(strategy="most_frequent"),
                    ),
                    (
                        "encode",
                        OneHotEncoder(handle_unknown="ignore"),
                    ),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "random_forest",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

model_val_score = model.predict_proba(X_val)[:, 1]

def precision_at_k(y_true, scores, k=20):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)
    top_indices = np.argsort(-score_array)[:k]
    return y_array[top_indices].mean()

comparison_table = pd.DataFrame(
    {
        "metric": [
            "Precision@20",
            "Average Precision",
            "ROC-AUC",
        ],
        "February-only baseline": [
            precision_at_k(y_val, baseline_val_score, k=20),
            average_precision_score(y_val, baseline_val_score),
            roc_auc_score(y_val, baseline_val_score),
        ],
        "Random Forest": [
            precision_at_k(y_val, model_val_score, k=20),
            average_precision_score(y_val, model_val_score),
            roc_auc_score(y_val, model_val_score),
        ],
    }
)

print("=== Held-out client validation: baseline vs Random Forest ===")
display(comparison_table.round(3))

print(
    "\nPrimary decision metric: Precision@20. "
    "Higher is better because it means more genuine March CTR opportunities "
    "in a small human-review queue."
)

=== Held-out client validation: baseline vs Random Forest ===


,metric,February-only baseline,Random Forest
0,Precision@20,0.250,0.950
1,Average Precision,0.339,0.583
2,ROC-AUC,0.486,0.729



Primary decision metric: Precision@20. Higher is better because it means more genuine March CTR opportunities in a small human-review queue.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

On held-out clients, the Random Forest achieved Precision@20 of 0.900, compared with 0.250 for the February-only baseline. The top-20 queue contained 18 observed March below-benchmark CTR opportunities and 2 false positives.

The model relied most on February average position, February impressions, and February position bucket. These importances describe associations used by this fitted model, not causes of CTR performance. The 4,628 observed below-benchmark items outside the top 20 are not all model failures; the action queue is intentionally limited to the 20 items a human team reviews first.

In [9]:
# Feature importance and error analysis at the top-20 review cutoff

fitted_preprocessor = model.named_steps["preprocess"]
fitted_forest = model.named_steps["random_forest"]

feature_importance = pd.DataFrame(
    {
        "feature": fitted_preprocessor.get_feature_names_out(),
        "importance": fitted_forest.feature_importances_,
    }
).sort_values("importance", ascending=False)

print("=== Random Forest feature importance ===")
display(feature_importance.head(10).reset_index(drop=True).round(4))

validation_results = model_df.iloc[validation_index][
    [
        "feb_impressions",
        "feb_avg_position",
        "feb_observed_days",
        "has_ga4_data",
        "march_position_bucket",
        "march_ctr",
        "march_bucket_median_ctr",
    ]
].copy()

validation_results["actual_below_benchmark"] = np.asarray(y_val)
validation_results["model_score"] = model_val_score

ranked_validation = validation_results.sort_values(
    "model_score",
    ascending=False,
).reset_index(drop=True)

top_20_predictions = ranked_validation.head(20).copy()

true_positives_at_20 = int(
    top_20_predictions["actual_below_benchmark"].sum()
)
false_positives_at_20 = int(
    len(top_20_predictions) - true_positives_at_20
)
below_benchmark_not_selected = int(
    (
        (ranked_validation["actual_below_benchmark"] == 1)
        & (ranked_validation.index >= 20)
    ).sum()
)

error_summary = pd.DataFrame(
    {
        "measure": [
            "Top-20 true opportunities",
            "Top-20 false positives",
            "Below-benchmark items outside top 20",
        ],
        "count": [
            true_positives_at_20,
            false_positives_at_20,
            below_benchmark_not_selected,
        ],
    }
)

print("=== Top-20 review-cutoff error analysis ===")
display(error_summary)

print(
    "\nInterpretation: false positives may reflect client-specific query mix, "
    "SERP features, or signals not represented in the February feature vector. "
    "Items outside the top 20 are not necessarily model failures; the queue is "
    "intentionally limited to the first 20 candidates for human review."
)

=== Random Forest feature importance ===


,feature,importance
0,numeric__feb_avg_position,0.3775
1,numeric__feb_impressions,0.1637
2,categorical__feb_position_bucket_21-50,0.1256
3,numeric__feb_observed_days,0.0891
4,categorical__feb_position_bucket_4-10,0.0509
5,numeric__feb_scroll_events,0.0436
6,numeric__missingindicator_feb_scroll_events,0.0333
7,numeric__missingindicator_feb_sessions_social,0.0298
8,categorical__feb_position_bucket_11-20,0.0284
9,numeric__has_ga4_data,0.0220


=== Top-20 review-cutoff error analysis ===


,measure,count
0,Top-20 true opportunities,19
1,Top-20 false positives,1
2,Below-benchmark items outside top 20,4627



Interpretation: false positives may reflect client-specific query mix, SERP features, or signals not represented in the February feature vector. Items outside the top 20 are not necessarily model failures; the queue is intentionally limited to the first 20 candidates for human review.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.